# Composite stress tests

Stack **rates**, **credit**, and **FX** using **`compose_scenarios`**, inspect **historical replay** templates, and see how **priority** orders specs when merging.

## Concept

- **Composition**: `compose_scenarios` accepts a typed **list of `ScenarioSpec` objects** and returns a **single** spec whose operations are merged per engine rules.
- **Authoring**: each per-factor spec is built from typed **`OperationSpec`** operations — `curve_parallel_bp` with `CurveKind.discount()` / `CurveKind.par_cds()` for rates and credit, `market_fx_pct` for FX.
- **Priority**: each spec carries an integer **`priority`** (lower runs earlier in composition—see merged ordering in printed JSON).
- **Historical templates**: built-ins such as **`gfc_2008`**, **`covid_2020`**, **`rate_shock_2022`** bundle multiple risk factors; **`list_template_components`** exposes rates/credit/vol/FX building blocks.
- **FX state**: the pair must exist in the typed `FxMatrix` before the market is serialized; scenario shocks then preserve the same pair and policy state.


### Composed vs sequential application

**`compose_scenarios`** merges multiple `ScenarioSpec` values into **one** spec. Each input spec has an integer **`priority`**; the merged **`operations`** list reflects the engine’s ordering (**lower priority number ⇒ earlier in the merged stack**—print the numbered operations below to verify).

**Sequential** stress means calling **`apply_scenario`** on the **base** market, then again on the returned **`MarketContext`**, and so on—the timeline is whatever order you choose in code. **Composed** stress applies **all** merged operations in **one** call. Because discounting, credit, and FX steps **do not always commute**, the **terminal market** from “compose once” vs “apply three times” can differ; comparing the two is a useful sanity check.

In [ ]:
import json

from finstack_quant.scenarios import (
    CurveKind,
    OperationSpec,
    apply_scenario,
    ScenarioSpec,
    build_template_component,
    compose_scenarios,
    list_builtin_template_metadata,
    list_builtin_templates,
    list_template_components,
    validate_scenario_spec,
)
print("apply_scenario:", callable(apply_scenario), "build_template_component:", callable(build_template_component))

print("Historical / stress templates:", list_builtin_templates())
meta = list_builtin_template_metadata()
print("Metadata entries:", len(meta))

for tid in ("gfc_2008", "covid_2020", "rate_shock_2022"):
    if tid in list_builtin_templates():
        comps = list_template_components(tid)
        print(f"Template {tid} components:", comps)


DISCOUNT = CurveKind.discount()
PAR_CDS = CurveKind.par_cds()

# One spec per risk factor, each authored with the typed builders and tagged with a
# composition priority (lower runs earlier in the merged stack).
rates_spec = ScenarioSpec(
    "step_rates",
    [OperationSpec.curve_parallel_bp(DISCOUNT, "USD-OIS", 40.0)],
    "Rates +40bp",
    "",
    priority=10,
)
credit_spec = ScenarioSpec(
    "step_credit",
    [OperationSpec.curve_parallel_bp(PAR_CDS, "ACME-HZD", 70.0, "USD-OIS")],
    "Credit +70bp",
    "",
    priority=20,
)
fx_spec = ScenarioSpec(
    "step_fx",
    [OperationSpec.market_fx_pct("EUR", "USD", 3.0)],
    "FX +3% EURUSD",
    "",
    priority=5,
)

composed = compose_scenarios([rates_spec, credit_spec, fx_spec])
composed_json = composed.to_json()
comp_obj = json.loads(composed_json)
print("Composed scenario id:", comp_obj.get("id"))
print("Composed operations count:", len(comp_obj.get("operations", [])))
print("validate composed:", validate_scenario_spec(composed_json))
print("First 500 chars of composed:", composed_json[:500])
print("Composed operations (verify merge / priority ordering):")
for i, op in enumerate(comp_obj.get("operations", [])):
    tag = op.get("curve_id") or op.get("base") or op.get("curve_kind")
    print(f"  [{i}] kind={op.get('kind')!r} ref={tag!r}")


In [ ]:
from datetime import date

from _shared.analyst_book import build_market
from finstack_quant.core.currency import Currency
from finstack_quant.core.market_data import MarketContext

bd = date(2025, 1, 15)
AS_OF = str(bd)

# The shared analyst market already carries everything this stress needs: a USD-OIS
# discount curve, the ACME-HZD hazard curve calibrated from CDS quotes (a `par_cds`
# shock refits it to the shocked quotes, so it must come from calibration), and a
# typed `FxMatrix` holding EUR/USD. Note the FX pair must be present *before* the
# market is serialized, otherwise `market_fx_pct` has nothing to shock.
mc = build_market("base", bd)
base_eurusd = mc.fx.rate(Currency("EUR"), Currency("USD"), bd, "cashflow_date").rate
print(f"Market with typed FX quote; EUR/USD={base_eurusd}")

# Multi-step composition with explicit priorities: FX 5, rates 10, credit 20
# (lower number runs earlier in the merged stack).
rates = ScenarioSpec(
    "r",
    [OperationSpec.curve_parallel_bp(DISCOUNT, "USD-OIS", 35.0)],
    "Rates +35bp",
    "",
    10,
)
credit = ScenarioSpec(
    "c",
    [OperationSpec.curve_parallel_bp(PAR_CDS, "ACME-HZD", 60.0, "USD-OIS")],
    "Credit +60bp",
    "",
    20,
)
fx = ScenarioSpec(
    "f",
    [OperationSpec.market_fx_pct("EUR", "USD", 4.0)],
    "FX +4% EURUSD",
    "",
    5,
)
composed = compose_scenarios([rates, credit, fx])
comp2 = json.loads(composed.to_json())
print("Composed spec priorities embedded; merged id:", comp2["id"])
print("Merged operation stack:")
for i, op in enumerate(comp2.get("operations", [])):
    tag = op.get("curve_id") or op.get("base") or op.get("curve_kind")
    print(f"  [{i}] kind={op.get('kind')!r} ref={tag!r}")

res = apply_scenario(composed, mc, AS_OF)
print("operations_applied:", res.report.operations_applied, "warnings:", res.report.warnings)

stressed = res.market
print("USD-OIS DF@1Y after:", stressed.get_discount("USD-OIS").df(1.0))
print("ACME-HZD survival@1Y after:", stressed.get_hazard("ACME-HZD").sp(1.0))
print("EUR/USD after stress:", stressed.fx.rate(Currency("EUR"), Currency("USD"), bd, "cashflow_date").rate)

def usd_ois_df_at_1y(market: MarketContext) -> float:
    return market.get_discount("USD-OIS").df(1.0)

def eurusd_rate(market: MarketContext) -> float:
    return market.fx.rate(Currency("EUR"), Currency("USD"), bd, "cashflow_date").rate

# Sequential timeline: FX, then credit, then rates. A `par_cds` refit first replays the
# hazard curve's calibration recipe on the input market's discount curve. Inside one
# composed call that is the pre-scenario USD-OIS, so credit may follow the rates shock;
# across separate calls a rate-shocked USD-OIS no longer reproduces the stored ACME-HZD
# curve and the refit is rejected, so the sequential credit step runs before rates.
sequential_market = mc
for label, spec in (("FX +4%", fx), ("Credit +60bp", credit), ("Rates +35bp", rates)):
    step = apply_scenario(spec, sequential_market, AS_OF)
    sequential_market = step.market
    print(f"Sequential {label}: operations_applied={step.report.operations_applied}")

print(
    "Sanity check — USD-OIS DF@1Y knot | composed vs sequential:",
    usd_ois_df_at_1y(res.market),
    usd_ois_df_at_1y(sequential_market),
)
print(
    "Sanity check — EUR/USD quote | composed vs sequential:",
    eurusd_rate(res.market),
    eurusd_rate(sequential_market),
)
# The composed refit solves the shocked CDS quotes on the rate-shocked USD-OIS; the
# sequential refit solved them on the base USD-OIS, so survival differs slightly.
print(
    "Sanity check — ACME-HZD survival@1Y | composed vs sequential:",
    stressed.get_hazard("ACME-HZD").sp(1.0),
    sequential_market.get_hazard("ACME-HZD").sp(1.0),
)

# Second pass: apply another small parallel rate shock to already stressed market (sequential application)
follow = ScenarioSpec(
    "f2",
    [OperationSpec.curve_parallel_bp(DISCOUNT, "USD-OIS", 10.0)],
    "Follow-on rates +10bp",
    "",
)
res2 = apply_scenario(follow, res.market, AS_OF)
print("Second apply operations_applied:", res2.report.operations_applied)
d2 = res2.market
print("USD-OIS DF@1Y after 2nd shock:", d2.get_discount("USD-OIS").df(1.0))


## Takeaways

- **`compose_scenarios`** merges multiple JSON specs—use it for **rates + credit + FX** (and more) in one application.
- Author each factor's operations with the typed **`OperationSpec`** builders (`curve_parallel_bp`, `market_fx_pct`); `ScenarioSpec` then wraps them with an id, labels, and a priority.
- **`priority`** on each spec controls merge ordering; print the composed JSON to confirm the final **`operations`** stack.
- **Historical templates** list via **`list_builtin_templates`**; drill into building blocks with **`list_template_components`**.
- **Chained application** (`apply` on an already stressed `MarketContext`) models **sequential** stress timelines; composition models **one combined** shock set.
- **FX shocks** need the pair present in the typed `FxMatrix` before the market is serialized — `_shared.analyst_book.build_market` sets up EUR/USD for you.